In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/co5420-human-activity-recognition/sample_submission.csv
/kaggle/input/competitions/co5420-human-activity-recognition/train.csv
/kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition/sample_submission.csv
/kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition/train.csv
/kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition/test.csv


In [2]:
# %% [markdown]
# # Human Activity Recognition (HAR) — Defensible & Measurable Solution
#
# **Goal**: Build a fully reproducible, group-aware, ablation-backed solution for the HAR Kaggle competition.
#
# ### Key Improvements & Validation Design:
# 1. **Group-Aware Validation Split (80/20 GroupShuffleSplit)**: Held-out subject split derived BEFORE model training.
# 2. **Mandatory Baselines**: Decision Tree & SVM (RBF), each lightly hyperparameter-searched via group-aware CV
#    on the training fold only, then scored on the held-out validation split.
# 3. **Validation-Backed Ablation Study**:
#    - Base NN Ensemble alone (Validation Macro F1)
#    - Base NN Ensemble + Self-Training / Pseudo-Labeling, using the SAME full ensemble mechanism as production
#      (Validation Macro F1)
#    - Base NN Ensemble + Temporal Sequence Smoothing (Validation Macro F1)
#    - COMBINED pipeline (pseudo-labeling + smoothing stacked together, exactly as Stage 3 will apply them)
#      (Validation Macro F1) — this, not the max of isolated ablations, is what gets reported as "final" and
#      is what determines the champion score, since it's the only number that matches what Stage 3 produces.
# 4. **Strict Column Assertions**: Loudly fails if expected sensor features are missing (no silent index fallbacks).
# 5. **Temporal Order Audit ON THE ACTUAL TARGET DATA**: Smoothing is applied to test.csv predictions in
#    production, so the contiguity audit checks test.csv subject ordering directly — NOT train.csv's validation
#    fold, which tells you nothing about test.csv's row order.
# 6. **Single Clean Deliverable**: Produces only `submission.csv` (`id,Activity`), with format assertions.

# %% [markdown]
# ## Cell 1: Environment & Dependency Setup

# %%
import os
import sys
import site
import time
import copy
import random
import numpy as np
import pandas as pd

# Windows PyTorch DLL path handling
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
try:
    for sp in site.getsitepackages():
        t_lib = os.path.join(sp, 'torch', 'lib')
        if os.path.exists(t_lib):
            os.environ['PATH'] = t_lib + os.pathsep + os.environ.get('PATH', '')
            if hasattr(os, 'add_dll_directory'):
                os.add_dll_directory(t_lib)
except Exception:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report

# Reproducibility seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENVIRONMENT] Active PyTorch device: {device} (CUDA available: {torch.cuda.is_available()})")

# %% [markdown]
# ## Cell 2: Dataset Loading & Column Validation

# %%
def find_data_file(filename):
    paths_to_check = [
        os.path.join('.', filename),
        os.path.join('co5420-human-activity-recognition', filename),
        os.path.join('..', filename),
        os.path.join('/kaggle/input/co5420-human-activity-recognition', filename),
        os.path.join('/kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition', filename),
        os.path.join('/kaggle/input/competitions/co5420-human-activity-recognition', filename)
    ]
    for p in paths_to_check:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not locate '{filename}' in search paths.")

train_path = find_data_file('train.csv')
test_path  = find_data_file('test.csv')

print(f"Loading data:\n  - Train: {train_path}\n  - Test:  {test_path}")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

# Ensure required columns exist
assert "Activity" in train_df.columns, "Missing required target column 'Activity' in train.csv!"
assert "subject" in train_df.columns,  "Missing required grouping column 'subject' in train.csv!"
assert "subject" in test_df.columns,   "Missing required grouping column 'subject' in test.csv!"

raw_feature_cols = [c for c in train_df.columns if c not in ["Activity", "id", "subject"]]
print(f"Raw Sensor Features: {len(raw_feature_cols)}")

# %% [markdown]
# ## Cell 3: Defensible Feature Engineering (With Explicit Assertions)

# %%
REQUIRED_PHYSICS_COLS = [
    'tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
    'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z'
]

def engineer_defensible_features(df, feature_cols):
    # Verify required physics columns exist — loud error if missing
    for col in REQUIRED_PHYSICS_COLS:
        assert col in df.columns, f"Required feature column '{col}' missing from dataset! Aborting feature engineering."

    X = df[feature_cols].copy()

    bx = df['tBodyAcc-mean()-X']
    by = df['tBodyAcc-mean()-Y']
    bz = df['tBodyAcc-mean()-Z']

    gx = df['tGravityAcc-mean()-X']
    gy = df['tGravityAcc-mean()-Y']
    gz = df['tGravityAcc-mean()-Z']

    bmag = np.sqrt(bx**2 + by**2 + bz**2)
    gmag = np.sqrt(gx**2 + gy**2 + gz**2)

    dot_prod  = (bx * gx + by * gy + bz * gz)
    cos_angle = dot_prod / (bmag * gmag + 1e-6)

    X['feat_body_grav_ratio']   = bmag / (gmag + 1e-6)
    X['feat_body_mag']          = bmag
    X['feat_grav_mag']          = gmag
    X['feat_body_grav_product'] = bmag * gmag
    X['feat_dot_prod']          = dot_prod
    X['feat_cos_angle']         = cos_angle

    # Domain EDA interaction features (if available)
    if 'angle(Y,gravityMean)' in df and 'tGravityAcc-mean()-Y' in df:
        X['feat_y_angle_ratio'] = df['angle(Y,gravityMean)'] / (df['tGravityAcc-mean()-Y'].abs() + 1e-6)
    if 'tBodyAccJerkMag-std()' in df and 'tBodyAccMag-std()' in df:
        X['feat_stair_jerk_ratio'] = df['tBodyAccJerkMag-std()'] / (df['tBodyAccMag-std()'].abs() + 1e-6)

    return X.values

X_train_engineered = engineer_defensible_features(train_df, raw_feature_cols)
X_test_engineered  = engineer_defensible_features(test_df,  raw_feature_cols)

label_encoder = LabelEncoder()
y_train_all = label_encoder.fit_transform(train_df["Activity"])
subjects_train_all = train_df["subject"].values
subjects_test = test_df["subject"].values

NUM_CLASSES = len(label_encoder.classes_)
INPUT_DIM   = X_train_engineered.shape[1]

print(f"Engineered Feature Dimension: {INPUT_DIM} features | Classes ({NUM_CLASSES}): {list(label_encoder.classes_)}")

# %% [markdown]
# ## Cell 4: Group-Aware Train / Validation Split (80/20 Subject Split)

# %%
# Group-aware split: validation set consists of ~20% UNSEEN SUBJECTS from train.csv
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(gss.split(X_train_engineered, y_train_all, groups=subjects_train_all))

X_tr_raw, X_val_raw = X_train_engineered[train_idx], X_train_engineered[val_idx]
y_tr, y_val         = y_train_all[train_idx], y_train_all[val_idx]
subj_tr, subj_val   = subjects_train_all[train_idx], subjects_train_all[val_idx]

print(f"Group Split Summary:")
print(f"  - Train Fold      : {len(train_idx)} samples ({len(np.unique(subj_tr))} subjects: {sorted(np.unique(subj_tr))})")
print(f"  - Validation Fold : {len(val_idx)} samples ({len(np.unique(subj_val))} subjects: {sorted(np.unique(subj_val))})")

# Per-subject feature normalization
def per_subject_normalize(X, subjects):
    X_norm = np.zeros_like(X, dtype=np.float64)
    for s in np.unique(subjects):
        mask = (subjects == s)
        mean = X[mask].mean(axis=0)
        std  = X[mask].std(axis=0) + 1e-8
        X_norm[mask] = (X[mask] - mean) / std
    return X_norm.astype(np.float32)

X_tr_norm  = per_subject_normalize(X_tr_raw, subj_tr)
X_val_norm = per_subject_normalize(X_val_raw, subj_val)
X_test_norm = per_subject_normalize(X_test_engineered, subjects_test)

# Scale features using RobustScaler fitted ONLY on training split
scaler = RobustScaler()
X_tr_scaled  = scaler.fit_transform(X_tr_norm)
X_val_scaled = scaler.transform(X_val_norm)

# Full training set scaled for final model fitting
X_train_all_norm   = per_subject_normalize(X_train_engineered, subjects_train_all)
scaler_full        = RobustScaler()
X_train_all_scaled = scaler_full.fit_transform(X_train_all_norm)
X_test_scaled      = scaler_full.transform(X_test_norm)

# %% [markdown]
# ## Cell 5: Mandatory Baselines (Decision Tree & SVM) — Lightly Tuned via Group-Aware CV
#
# Both baselines are searched over a small grid using GroupKFold on the TRAINING FOLD ONLY
# (never touching X_val/y_val), so the validation score reported below is a genuinely unseen
# evaluation of the tuned model — not a number produced by a fixed, arbitrary hyperparameter guess.

# %%
print("\n" + "="*60)
print("STAGE 1: MANDATORY BASELINE MODELS (Group-Aware Tuning + Validation)")
print("="*60)

# GroupKFold for tuning within the training fold (subject-aware, no leakage across subjects)
N_TUNING_FOLDS = 3
gkf_tune = GroupKFold(n_splits=N_TUNING_FOLDS)

# 1. Decision Tree Baseline — small depth/leaf grid to avoid an unconstrained,
#    badly-overfit tree acting as a strawman baseline.
dt_param_grid = {
    "max_depth": [8, 12, 16, 20, None],
    "min_samples_leaf": [1, 3, 5, 10],
}
dt_search = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid=dt_param_grid,
    scoring="f1_macro",
    cv=gkf_tune,
    n_jobs=-1,
)
dt_search.fit(X_tr_scaled, y_tr, groups=subj_tr)
dt_model = dt_search.best_estimator_
dt_val_preds = dt_model.predict(X_val_scaled)
dt_val_f1    = f1_score(y_val, dt_val_preds, average='macro')

print(f"\n[BASELINE 1] Decision Tree Classifier")
print(f"  Best Params (via GroupKFold CV on train fold): {dt_search.best_params_}")
print(f"  Validation Macro F1: {dt_val_f1:.4f}")
print("  Per-Class F1 Breakdown:")
for cls_idx, f1 in enumerate(f1_score(y_val, dt_val_preds, average=None)):
    print(f"    {label_encoder.classes_[cls_idx]:<22}: {f1:.4f}")

# 2. Support Vector Machine (SVM RBF) Baseline — small C/gamma grid instead of a fixed guess.
svm_param_grid = {
    "C": [1.0, 10.0, 50.0],
    "gamma": ["scale", 0.01, 0.001],
}
svm_search = GridSearchCV(
    SVC(kernel="rbf", probability=True, random_state=SEED),
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=gkf_tune,
    n_jobs=-1,
)
svm_search.fit(X_tr_scaled, y_tr, groups=subj_tr)
svm_model = svm_search.best_estimator_
svm_val_preds = svm_model.predict(X_val_scaled)
svm_val_f1    = f1_score(y_val, svm_val_preds, average='macro')

print(f"\n[BASELINE 2] Support Vector Machine (SVM RBF)")
print(f"  Best Params (via GroupKFold CV on train fold): {svm_search.best_params_}")
print(f"  Validation Macro F1: {svm_val_f1:.4f}")
print("  Per-Class F1 Breakdown:")
for cls_idx, f1 in enumerate(f1_score(y_val, svm_val_preds, average=None)):
    print(f"    {label_encoder.classes_[cls_idx]:<22}: {f1:.4f}")

# %% [markdown]
# ## Cell 6: PyTorch Deep Neural Network Ensemble Architecture

# %%
class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_blocks=2, dropout=0.35):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.out_proj = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = self.in_proj(x)
        for block in self.blocks: x = block(x)
        return self.out_proj(x)

class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes, dropout, act_type='gelu'):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       Swish() if act_type == 'swish' else nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

MODEL_CONFIGS = [
    {"type": "res", "name": "ResMLP_512",   "hidden": 512,  "blocks": 2, "dropout": 0.35},
    {"type": "res", "name": "ResMLP_768",   "hidden": 768,  "blocks": 3, "dropout": 0.30},
    {"type": "mlp", "name": "512_256_128", "hidden_sizes": [512, 256, 128], "dropout": 0.40, "act": "gelu"},
    {"type": "mlp", "name": "1024_512_256", "hidden_sizes": [1024, 512, 256], "dropout": 0.45, "act": "swish"},
    {"type": "mlp", "name": "512_128",      "hidden_sizes": [512, 128],       "dropout": 0.38, "act": "swish"}
]

SEEDS = [42, 43, 44]  # 5 configs x 3 seeds = 15 NN models
BATCH_SIZE = 64
LR = 1e-3
LABEL_SMOOTHING = 0.04

def make_loader(X, y, seed):
    gen = torch.Generator().manual_seed(seed)
    ds  = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen)

def train_nn_model(X_tr_, y_tr_, class_weights, cfg, epochs, seed, mixup_alpha=0.15):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    if cfg["type"] == "res":
        model = ResMLP(INPUT_DIM, cfg["hidden"], NUM_CLASSES, num_blocks=cfg["blocks"], dropout=cfg["dropout"]).to(device)
    else:
        model = DeepMLP(INPUT_DIM, cfg["hidden_sizes"], NUM_CLASSES, cfg["dropout"], act_type=cfg.get("act", "gelu")).to(device)

    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

    loader = make_loader(X_tr_, y_tr_, seed)

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            xb = xb + torch.randn_like(xb) * 0.008

            if mixup_alpha > 0 and np.random.rand() < 0.4:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                index = torch.randperm(xb.size(0)).to(device)
                xb_mixed = lam * xb + (1 - lam) * xb[index]
                yb_a, yb_b = yb, yb[index]
                opt.zero_grad()
                out = model(xb_mixed)
                loss = lam * criterion(out, yb_a) + (1 - lam) * criterion(out, yb_b)
            else:
                opt.zero_grad()
                out = model(xb)
                loss = criterion(out, yb)

            loss.backward()
            opt.step()
        sched.step()

    return model

def train_ensemble(X_tr_, y_tr_, class_weights, epochs, seed_offset=0):
    """Trains the full MODEL_CONFIGS x SEEDS ensemble. Factored out so every stage
    (base ablation, pseudo-labeling simulation, and final production training) uses
    the EXACT same mechanism — no more mismatched fidelity between a validation
    ablation and what actually runs on test.csv."""
    models = []
    for cfg in MODEL_CONFIGS:
        for seed in SEEDS:
            m = train_nn_model(X_tr_, y_tr_, class_weights, cfg, epochs=epochs, seed=seed + seed_offset)
            models.append(m)
    return models

def adabn_predict_proba(trained_model, X_subj_scaled):
    model = copy.deepcopy(trained_model)
    for module in model.modules():
        if isinstance(module, nn.BatchNorm1d):
            module.momentum = 1.0

    X_subj_t = torch.tensor(X_subj_scaled, dtype=torch.float32).to(device)
    model.train()
    with torch.no_grad():
        _ = model(X_subj_t)
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(X_subj_t), dim=1).cpu().numpy()

def ensemble_predict_proba_by_subject(models, X_scaled, subjects):
    """AdaBN ensemble prediction, applied per-subject (as production does)."""
    probs = np.zeros((len(X_scaled), NUM_CLASSES))
    for s in np.unique(subjects):
        idx = np.where(subjects == s)[0]
        s_probs = np.zeros((len(idx), NUM_CLASSES))
        for m in models:
            s_probs += adabn_predict_proba(m, X_scaled[idx])
        probs[idx] = s_probs / len(models)
    return probs

def temporal_smooth(probs, subjects, window_size=3):
    smoothed = np.zeros_like(probs)
    kernel   = np.array([0.15, 0.70, 0.15])
    pad      = window_size // 2
    for s in np.unique(subjects):
        idx = np.where(subjects == s)[0]
        sp  = probs[idx]
        padded = np.pad(sp, ((pad, pad), (0, 0)), mode='edge')
        sm  = np.zeros_like(sp)
        for t in range(len(sp)):
            sm[t] = np.sum(padded[t:t+window_size] * kernel[:, None], axis=0)
        smoothed[idx] = sm
    return smoothed

def class_weights_for(y):
    counts = np.bincount(y, minlength=NUM_CLASSES)
    return torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32).to(device)

# %% [markdown]
# ## Cell 7: Controlled Ablation Study on Held-Out Validation Split
#
# Every stage here uses `train_ensemble(...)` / `ensemble_predict_proba_by_subject(...)` — the
# SAME functions Stage 3 uses on test.csv — so these validation numbers are honest estimates of
# what the equivalent production stage will do, not a lighter-weight stand-in.

# %%
print("\n" + "="*60)
print("STAGE 2: ABLATION STUDY ON HELD-OUT VALIDATION SPLIT")
print("="*60)

cw_tr = class_weights_for(y_tr)

# --- Ablation A: Base NN Ensemble Alone ---
print("\n[ABLATION A] Base NN Ensemble Alone (No Pseudo-Labeling, No Smoothing)")
start_nn = time.time()
base_nn_models = train_ensemble(X_tr_scaled, y_tr, cw_tr, epochs=30, seed_offset=0)
val_base_probs = ensemble_predict_proba_by_subject(base_nn_models, X_val_scaled, subj_val)
ablation_a_f1 = f1_score(y_val, val_base_probs.argmax(axis=1), average='macro')
print(f"  -> Base NN Ensemble Validation Macro F1: {ablation_a_f1:.4f} (trained in {time.time()-start_nn:.1f}s)")

# --- Ablation B: + Fair Validation-Safe Pseudo-Labeling Simulation ---
# Uses the SAME full ensemble mechanism as production (train_ensemble / ensemble_predict_proba_by_subject),
# not a single lightweight MLP stand-in, so this ablation is a faithful proxy for what Stage 3 will do.
print("\n[ABLATION B] + Self-Training / Pseudo-Labeling Simulation (full ensemble, matches production)")
split_subjs = np.unique(subj_tr)
n_seed = int(len(split_subjs) * 0.70)
seed_subjs, pool_subjs = split_subjs[:n_seed], split_subjs[n_seed:]

seed_mask = np.isin(subj_tr, seed_subjs)
pool_mask = np.isin(subj_tr, pool_subjs)

cw_seed = class_weights_for(y_tr[seed_mask])
seed_ensemble = train_ensemble(X_tr_scaled[seed_mask], y_tr[seed_mask], cw_seed, epochs=30, seed_offset=200)

pool_probs = ensemble_predict_proba_by_subject(seed_ensemble, X_tr_scaled[pool_mask], subj_tr[pool_mask])

pseudo_mask_tr = pool_probs.max(axis=1) >= 0.95
n_pseudo_tr    = pseudo_mask_tr.sum()

if n_pseudo_tr > 0:
    X_comb_sim = np.vstack([X_tr_scaled[seed_mask], X_tr_scaled[pool_mask][pseudo_mask_tr]])
    y_comb_sim = np.concatenate([y_tr[seed_mask], pool_probs[pseudo_mask_tr].argmax(axis=1)])

    cw_sim = class_weights_for(y_comb_sim)
    pseudo_ensemble = train_ensemble(X_comb_sim, y_comb_sim, cw_sim, epochs=30, seed_offset=300)

    val_pseudo_probs = ensemble_predict_proba_by_subject(pseudo_ensemble, X_val_scaled, subj_val)
    ablation_b_f1 = f1_score(y_val, val_pseudo_probs.argmax(axis=1), average='macro')
    print(f"  -> Extracted {n_pseudo_tr} pseudo-labels from training pool.")
    print(f"  -> Validation Macro F1 with Pseudo-Labeling: {ablation_b_f1:.4f}")
else:
    ablation_b_f1 = ablation_a_f1
    print("  -> No pseudo-labels exceeded threshold in simulation.")

# --- Ablation C: + Temporal Sequence Smoothing (applied to the base ensemble's val probs) ---
print("\n[ABLATION C] + Temporal Sequence Probability Smoothing")
val_smoothed_probs = temporal_smooth(val_base_probs, subj_val, window_size=3)
ablation_c_f1      = f1_score(y_val, val_smoothed_probs.argmax(axis=1), average='macro')
print(f"  -> Validation Macro F1 with Temporal Smoothing (train-fold data): {ablation_c_f1:.4f}")
print("  -> NOTE: this number only tells us whether the smoothing MATH helps IF rows are")
print("     contiguous. Whether it's safe to apply in production depends on TEST.CSV's own")
print("     row ordering, audited separately below (not on this train-fold validation split).")

# --- Temporal Sequence Order Audit — ON TEST.CSV, THE ACTUAL SMOOTHING TARGET ---
print("\n" + "-"*60)
print("TEMPORAL SEQUENCE ORDER AUDIT (on test.csv, since that's what smoothing actually touches):")
sequence_audit_test = {}
for s in np.unique(subjects_test):
    s_indices = np.where(subjects_test == s)[0]
    is_contiguous = np.all(np.diff(s_indices) == 1) if len(s_indices) > 1 else True
    sequence_audit_test[s] = (len(s_indices), is_contiguous)

all_contiguous_test = all(c[1] for c in sequence_audit_test.values())
print(f"  Subject contiguous row index check (test.csv): {sequence_audit_test}")
print(f"  Are test.csv subject rows contiguous? {all_contiguous_test}")
if not all_contiguous_test:
    print("  -> test.csv rows are NOT contiguous per subject. Applying a positional smoothing")
    print("     kernel would average across unrelated/shuffled rows and could corrupt")
    print("     predictions. Temporal smoothing will be DISABLED regardless of the ablation")
    print("     C score above.")

# Decision Flags (individual)
ENABLE_PSEUDO_LABELING    = (ablation_b_f1 > ablation_a_f1)
ENABLE_TEMPORAL_SMOOTHING = (ablation_c_f1 > ablation_a_f1) and all_contiguous_test

# --- Ablation D: COMBINED pipeline, stacked in the exact order Stage 3 will use it ---
# This is the only number that actually matches what gets written to submission.csv when both
# stages are enabled — reporting max(A, B, C) instead (as before) would silently misrepresent
# the final pipeline's true validation performance whenever both stages are enabled together.
print("\n[ABLATION D] Combined Pipeline — stages stacked in production order")
if ENABLE_PSEUDO_LABELING:
    combined_probs = val_pseudo_probs.copy()
else:
    combined_probs = val_base_probs.copy()

if ENABLE_TEMPORAL_SMOOTHING:
    # smoothing here is only a proxy check using the train-fold validation split's own
    # subject contiguity, purely to report a comparable ablation number; the production
    # ENABLE_TEMPORAL_SMOOTHING flag is already gated on test.csv's real contiguity above.
    combined_probs = temporal_smooth(combined_probs, subj_val, window_size=3)

ablation_d_f1 = f1_score(y_val, combined_probs.argmax(axis=1), average='macro')
print(f"  -> Combined pipeline Validation Macro F1: {ablation_d_f1:.4f}")
print(f"     (Pseudo-labeling: {'ON' if ENABLE_PSEUDO_LABELING else 'OFF'}, "
      f"Temporal smoothing: {'ON' if ENABLE_TEMPORAL_SMOOTHING else 'OFF'})")

print("\n" + "="*60)
print("ABLATION SUMMARY TABLE")
print("="*60)
print(f"  1. Base NN Ensemble Alone      : F1 = {ablation_a_f1:.4f} [Baseline Score]")
print(f"  2. + Pseudo-Labeling (isolated) : F1 = {ablation_b_f1:.4f} [{'ENABLED ✅' if ENABLE_PSEUDO_LABELING else 'DISABLED ❌ (did not improve validation)'}]")
print(f"  3. + Temporal Smoothing (isolated): F1 = {ablation_c_f1:.4f} [{'ENABLED ✅' if ENABLE_TEMPORAL_SMOOTHING else 'DISABLED ❌ (did not improve validation, or test.csv is non-contiguous)'}]")
print(f"  4. COMBINED (production order)  : F1 = {ablation_d_f1:.4f} [This IS what Stage 3 will produce]")
print("="*60)

# %% [markdown]
# ## Cell 7.5: Hierarchical / Cascaded Classifier for the Two Known-Hard Pairs
#
# The per-class F1 breakdowns above (and every confusion matrix in this project) point at two
# specific confusions: SITTING vs STANDING, and WALKING_UPSTAIRS vs WALKING_DOWNSTAIRS. Both are
# hard for a flat 6-way classifier because the dominant signal (raw accel/gyro magnitude) barely
# differs between each pair — the features that DO separate them (gravity-angle features for
# sitting/standing; jerk-asymmetry features for the stairs pair) are a small minority among 561+
# inputs and get diluted in a flat classifier.
#
# This cascade is a real trained-and-cross-validated pipeline, NOT a hand-coded threshold rule:
#   Stage 1: a tuned classifier predicts one of 4 COARSE groups
#   Stage 2: for the two ambiguous coarse groups, a dedicated binary classifier (trained ONLY on
#            that subgroup) makes the final call
# Every stage is validated on the SAME held-out group split as everything else in this script, and
# the cascade is only used for the final submission if it beats the flat pipeline's honest
# validation score (Ablation D). If it doesn't help, we say so and keep the flat pipeline.

# %%
print("\n" + "="*60)
print("STAGE 2.5: HIERARCHICAL / CASCADED CLASSIFIER (Hard-Pair Focus)")
print("="*60)

GROUP_STATIC_SIT_STAND = 0   # SITTING, STANDING
GROUP_LAYING           = 1   # LAYING
GROUP_WALK_UP_DOWN     = 2   # WALKING_UPSTAIRS, WALKING_DOWNSTAIRS
GROUP_WALKING          = 3   # WALKING
GROUP_NAMES = {
    GROUP_STATIC_SIT_STAND: "STATIC_SIT_STAND",
    GROUP_LAYING: "LAYING",
    GROUP_WALK_UP_DOWN: "WALK_UP_DOWN",
    GROUP_WALKING: "WALKING",
}

def activity_name_to_group(name):
    if name in ("SITTING", "STANDING"):
        return GROUP_STATIC_SIT_STAND
    if name == "LAYING":
        return GROUP_LAYING
    if name in ("WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS"):
        return GROUP_WALK_UP_DOWN
    if name == "WALKING":
        return GROUP_WALKING
    raise ValueError(f"Unrecognized activity label: {name}")

# Map every encoded label id -> its coarse group id, once
LABEL_ID_TO_GROUP = np.array([
    activity_name_to_group(name) for name in label_encoder.classes_
])

def to_coarse(y_encoded):
    return LABEL_ID_TO_GROUP[y_encoded]

y_tr_coarse  = to_coarse(y_tr)
y_val_coarse = to_coarse(y_val)

# --- Stage 1: Coarse 4-group classifier (tuned SVM, same style of tuning as the baselines) ---
print("\n[CASCADE STAGE 1] Coarse 4-group classifier")
coarse_param_grid = {"C": [1.0, 10.0, 50.0], "gamma": ["scale", 0.01, 0.001]}
coarse_search = GridSearchCV(
    SVC(kernel="rbf", probability=True, random_state=SEED),
    param_grid=coarse_param_grid,
    scoring="f1_macro",
    cv=gkf_tune,
    n_jobs=-1,
)
coarse_search.fit(X_tr_scaled, y_tr_coarse, groups=subj_tr)
coarse_model = coarse_search.best_estimator_
coarse_val_f1 = f1_score(y_val_coarse, coarse_model.predict(X_val_scaled), average="macro")
print(f"  Best Params: {coarse_search.best_params_}")
print(f"  Validation Macro F1 (4 coarse groups): {coarse_val_f1:.4f}")

# --- Stage 2A: SITTING vs STANDING sub-classifier (small tuned Decision Tree, interpretable) ---
print("\n[CASCADE STAGE 2A] SITTING vs STANDING sub-classifier")
sit_stand_mask_tr = (y_tr_coarse == GROUP_STATIC_SIT_STAND)
sub_param_grid = {"max_depth": [4, 6, 8, 12, None], "min_samples_leaf": [1, 3, 5]}

sitstand_search = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid=sub_param_grid,
    scoring="f1_macro",
    cv=GroupKFold(n_splits=min(N_TUNING_FOLDS, len(np.unique(subj_tr[sit_stand_mask_tr])))),
    n_jobs=-1,
)
sitstand_search.fit(X_tr_scaled[sit_stand_mask_tr], y_tr[sit_stand_mask_tr], groups=subj_tr[sit_stand_mask_tr])
sitstand_model = sitstand_search.best_estimator_

sit_stand_mask_val = (y_val_coarse == GROUP_STATIC_SIT_STAND)
if sit_stand_mask_val.sum() > 0:
    sitstand_val_f1 = f1_score(
        y_val[sit_stand_mask_val], sitstand_model.predict(X_val_scaled[sit_stand_mask_val]), average="macro"
    )
else:
    sitstand_val_f1 = float("nan")
print(f"  Best Params: {sitstand_search.best_params_}")
print(f"  Binary Validation Macro F1 (within STATIC_SIT_STAND rows): {sitstand_val_f1:.4f}")

top_feat_idx = np.argsort(sitstand_model.feature_importances_)[::-1][:10]
sub_feature_names = raw_feature_cols + [c for c in
    ["feat_body_grav_ratio","feat_body_mag","feat_grav_mag","feat_body_grav_product",
     "feat_dot_prod","feat_cos_angle","feat_y_angle_ratio","feat_stair_jerk_ratio"]
    if True][:X_tr_scaled.shape[1]]
print("  Top 10 features driving SITTING vs STANDING split:")
for i in top_feat_idx:
    name = sub_feature_names[i] if i < len(sub_feature_names) else f"feature_{i}"
    print(f"    {name:<35}: importance={sitstand_model.feature_importances_[i]:.4f}")

# --- Stage 2B: WALKING_UPSTAIRS vs WALKING_DOWNSTAIRS sub-classifier ---
print("\n[CASCADE STAGE 2B] WALKING_UPSTAIRS vs WALKING_DOWNSTAIRS sub-classifier")
updown_mask_tr = (y_tr_coarse == GROUP_WALK_UP_DOWN)

updown_search = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid=sub_param_grid,
    scoring="f1_macro",
    cv=GroupKFold(n_splits=min(N_TUNING_FOLDS, len(np.unique(subj_tr[updown_mask_tr])))),
    n_jobs=-1,
)
updown_search.fit(X_tr_scaled[updown_mask_tr], y_tr[updown_mask_tr], groups=subj_tr[updown_mask_tr])
updown_model = updown_search.best_estimator_

updown_mask_val = (y_val_coarse == GROUP_WALK_UP_DOWN)
if updown_mask_val.sum() > 0:
    updown_val_f1 = f1_score(
        y_val[updown_mask_val], updown_model.predict(X_val_scaled[updown_mask_val]), average="macro"
    )
else:
    updown_val_f1 = float("nan")
print(f"  Best Params: {updown_search.best_params_}")
print(f"  Binary Validation Macro F1 (within WALK_UP_DOWN rows): {updown_val_f1:.4f}")

top_feat_idx_ud = np.argsort(updown_model.feature_importances_)[::-1][:10]
print("  Top 10 features driving UPSTAIRS vs DOWNSTAIRS split:")
for i in top_feat_idx_ud:
    name = sub_feature_names[i] if i < len(sub_feature_names) else f"feature_{i}"
    print(f"    {name:<35}: importance={updown_model.feature_importances_[i]:.4f}")

# --- Combine into final 6-class cascade prediction ---
def predict_cascade(X_scaled, coarse_clf, sitstand_clf, updown_clf):
    """Stage 1 routes to coarse group; Stage 2 resolves the two ambiguous groups.
    LAYING/WALKING pass through with the coarse-stage label directly (no ambiguity there)."""
    coarse_pred = coarse_clf.predict(X_scaled)
    final_pred = np.zeros(len(X_scaled), dtype=int)

    laying_id  = label_encoder.transform(["LAYING"])[0]
    walking_id = label_encoder.transform(["WALKING"])[0]

    m_static = (coarse_pred == GROUP_STATIC_SIT_STAND)
    m_laying = (coarse_pred == GROUP_LAYING)
    m_updown = (coarse_pred == GROUP_WALK_UP_DOWN)
    m_walk   = (coarse_pred == GROUP_WALKING)

    if m_static.sum() > 0:
        final_pred[m_static] = sitstand_clf.predict(X_scaled[m_static])
    final_pred[m_laying] = laying_id
    if m_updown.sum() > 0:
        final_pred[m_updown] = updown_clf.predict(X_scaled[m_updown])
    final_pred[m_walk] = walking_id
    return final_pred

cascade_val_preds = predict_cascade(X_val_scaled, coarse_model, sitstand_model, updown_model)
cascade_val_f1 = f1_score(y_val, cascade_val_preds, average="macro")

print("\n[CASCADE OVERALL] Full 6-class validation performance")
print(f"  Cascade Validation Macro F1  : {cascade_val_f1:.4f}")
print(f"  Flat Pipeline Validation Macro F1 (Ablation D, for comparison): {ablation_d_f1:.4f}")
print("  Per-class F1 (cascade) for the two hard pairs specifically:")
cascade_per_class_f1 = f1_score(y_val, cascade_val_preds, average=None)
for cls_name in ["SITTING", "STANDING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS"]:
    cls_idx = list(label_encoder.classes_).index(cls_name)
    print(f"    {cls_name:<20}: {cascade_per_class_f1[cls_idx]:.4f}")

USE_CASCADE_FOR_SUBMISSION = cascade_val_f1 > ablation_d_f1
print("\n" + "-"*60)
if USE_CASCADE_FOR_SUBMISSION:
    print(f"DECISION: Cascade WINS on validation ({cascade_val_f1:.4f} > {ablation_d_f1:.4f}).")
    print("  -> The final submission will use the hierarchical cascade instead of the flat pipeline.")
    print("  -> Tradeoff to note: the cascade requires training/maintaining 3 separate models")
    print("     (1 coarse + 2 binary) instead of one flat model, at added inference complexity.")
else:
    print(f"DECISION: Flat pipeline WINS or ties on validation ({ablation_d_f1:.4f} >= {cascade_val_f1:.4f}).")
    print("  -> The cascade did NOT demonstrably improve macro F1 here. This can happen when the")
    print("     flat ensemble already has enough capacity/regularization to implicitly learn the")
    print("     same separation the cascade targets explicitly. The final submission will use the")
    print("     flat pipeline; the cascade code above is kept for the report as a documented,")
    print("     honestly-negative result rather than silently discarded.")
print("-"*60)

# %% [markdown]
# ## Cell 8: Final Model Training & Test Set Predictions

# %%
print("\n" + "="*60)
print("STAGE 3: FINAL MODEL TRAINING ON FULL DATASET")
print("="*60)

cw_all = class_weights_for(y_train_all)

if USE_CASCADE_FOR_SUBMISSION:
    # ---- Retrain the winning cascade on the FULL labeled dataset ----
    print("Cascade won validation — retraining coarse + 2 binary sub-classifiers on full train.csv...")
    y_train_all_coarse = to_coarse(y_train_all)

    coarse_model_full = SVC(**coarse_search.best_params_, kernel="rbf", probability=True, random_state=SEED)
    coarse_model_full.fit(X_train_all_scaled, y_train_all_coarse)

    sit_stand_mask_full = (y_train_all_coarse == GROUP_STATIC_SIT_STAND)
    sitstand_model_full = DecisionTreeClassifier(**sitstand_search.best_params_, random_state=SEED)
    sitstand_model_full.fit(X_train_all_scaled[sit_stand_mask_full], y_train_all[sit_stand_mask_full])

    updown_mask_full = (y_train_all_coarse == GROUP_WALK_UP_DOWN)
    updown_model_full = DecisionTreeClassifier(**updown_search.best_params_, random_state=SEED)
    updown_model_full.fit(X_train_all_scaled[updown_mask_full], y_train_all[updown_mask_full])

    final_preds_numeric = predict_cascade(X_test_scaled, coarse_model_full, sitstand_model_full, updown_model_full)
    final_preds_labels = label_encoder.inverse_transform(final_preds_numeric)
    print(f"  Cascade retrained and applied to {len(test_df)} test rows.")

else:
    # ---- Flat pipeline: NN ensemble (+ pseudo-labeling / temporal smoothing if validated) ----
    print(f"Flat pipeline won validation — training final {len(MODEL_CONFIGS)*len(SEEDS)}-model NN ensemble on full train.csv...")
    start_final = time.time()
    final_base_models = train_ensemble(X_train_all_scaled, y_train_all, cw_all, epochs=35, seed_offset=0)
    final_test_probs = ensemble_predict_proba_by_subject(final_base_models, X_test_scaled, subjects_test)
    print(f"  Base ensemble trained in {time.time() - start_final:.1f}s")

    # Step 2: Conditional Pseudo-Labeling (Only if validated)
    if ENABLE_PSEUDO_LABELING:
        print("Applying validated Pseudo-Labeling to final model...")
        conf_test = final_test_probs.max(axis=1)
        pseudo_mask_test = conf_test >= 0.95
        print(f"  Adding {pseudo_mask_test.sum()} high-confidence test samples to training set.")

        X_comb_final = np.vstack([X_train_all_scaled, X_test_scaled[pseudo_mask_test]])
        y_comb_final = np.concatenate([y_train_all, final_test_probs[pseudo_mask_test].argmax(axis=1)])

        cw_comb_final = class_weights_for(y_comb_final)
        final_pseudo_models = train_ensemble(X_comb_final, y_comb_final, cw_comb_final, epochs=35, seed_offset=100)
        final_test_probs = ensemble_predict_proba_by_subject(final_pseudo_models, X_test_scaled, subjects_test)

    # Step 3: Conditional Temporal Smoothing (Only if validated AND test.csv confirmed contiguous)
    if ENABLE_TEMPORAL_SMOOTHING:
        print("Applying validated Temporal Smoothing to test predictions...")
        final_test_probs = temporal_smooth(final_test_probs, subjects_test, window_size=3)
    else:
        print("Temporal Smoothing disabled for the final submission (see audit/ablation above).")

    final_preds_numeric = final_test_probs.argmax(axis=1)
    final_preds_labels  = label_encoder.inverse_transform(final_preds_numeric)

test_ids = test_df["id"].values if "id" in test_df.columns else np.arange(1, len(test_df) + 1)
submission_df = pd.DataFrame({"id": test_ids, "Activity": final_preds_labels})

# --- Submission format assertions (cheap insurance against silent format bugs) ---
VALID_LABELS = set(label_encoder.classes_)
assert len(submission_df) == len(test_df), \
    f"Row count mismatch: submission has {len(submission_df)}, test.csv has {len(test_df)}"
assert submission_df["id"].isna().sum() == 0, "submission.csv contains missing ids!"
assert set(submission_df["Activity"].unique()).issubset(VALID_LABELS), \
    f"submission.csv contains invalid activity labels: {set(submission_df['Activity'].unique()) - VALID_LABELS}"
assert list(submission_df.columns) == ["id", "Activity"], \
    f"submission.csv has unexpected columns: {list(submission_df.columns)}"
print("\nSubmission format assertions passed: correct row count, no missing ids, "
      "only valid labels, exactly [id, Activity] columns.")

# Write ONLY submission.csv matching sample_submission.csv format exactly
submission_output_path = "submission_improved.csv"
submission_df.to_csv(submission_output_path, index=False)

print(f"\nSaved final submission file to '{submission_output_path}' ({len(submission_df)} rows).")

# %% [markdown]
# ## Cell 9: Final Summary Table & Champion Verification

# %%
print("\n" + "="*70)
print("FINAL SUMMARY TABLE & CHAMPION MODEL EVALUATION")
print("="*70)
print(f"{'Model / Stage Architecture':<38} | {'Validation Macro F1':>20} | {'Status':<12}")
print("-" * 75)
print(f"{'Decision Tree Baseline (tuned)':<38} | {dt_val_f1:>20.4f} | {'Baseline':<12}")
print(f"{'Support Vector Machine (tuned)':<38} | {svm_val_f1:>20.4f} | {'Baseline':<12}")
print(f"{'Base NN Ensemble (15 Models)':<38} | {ablation_a_f1:>20.4f} | {'Candidate':<12}")
print(f"{'+ Pseudo-Labeling (isolated)':<38} | {ablation_b_f1:>20.4f} | {'Candidate':<12}")
print(f"{'+ Temporal Smoothing (isolated)':<38} | {ablation_c_f1:>20.4f} | {'Candidate':<12}")
print(f"{'Flat Pipeline (combined, Ablation D)':<38} | {ablation_d_f1:>20.4f} | {'Candidate' if USE_CASCADE_FOR_SUBMISSION else 'CHAMPION 🏆':<12}")
print(f"{'Hierarchical Cascade (coarse + 2 sub)':<38} | {cascade_val_f1:>20.4f} | {'CHAMPION 🏆' if USE_CASCADE_FOR_SUBMISSION else 'Candidate':<12}")
print("="*70)
print(f"\nNote: the CHAMPION row above (validation score {cascade_val_f1 if USE_CASCADE_FOR_SUBMISSION else ablation_d_f1:.4f})")
print("is exactly what was retrained on the full dataset and applied to produce submission.csv —")
print("not the best of any isolated ablation.")

print(f"\nFinal Class Distribution in Test Predictions:")
print(submission_df['Activity'].value_counts())
print("\nPipeline execution completed cleanly and defensibly!")


[ENVIRONMENT] Active PyTorch device: cuda (CUDA available: True)
Loading data:
  - Train: /kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition/train.csv
  - Test:  /kaggle/input/competitions/co5420-human-activity-recognition/co5420-human-activity-recognition/test.csv
Train Shape: (7352, 563) | Test Shape: (2947, 563)
Raw Sensor Features: 561
Engineered Feature Dimension: 569 features | Classes (6): ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']
Group Split Summary:
  - Train Fold      : 5551 samples (16 subjects: [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(11), np.int64(14), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(26), np.int64(28), np.int64(29), np.int64(30)])
  - Validation Fold : 1801 samples (5 subjects: [np.int64(1), np.int64(3), np.int64(15), np.int64(25), np.int64(27)])

STAGE 1: MANDATORY BASELINE MODELS (Group-Aware Tuning + V